# Finetune

In [1]:
! pip install --upgrade transformers trl accelerate bitsandbytes peft datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 83.7 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.9/511.9 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.7/374.7 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 24.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.9/503.9 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.8/494.8 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.5/561.5 kB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 83.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 78.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 41.3 

In [2]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

In [3]:
from huggingface_hub import login

login(HF_TOKEN)

In [4]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
import torch

2025-08-09 15:15:17.368573: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1754752517.600043      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1754752517.662341      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [5]:
model_id = "Qwen/Qwen3-4B"
data_path = "/kaggle/input/unidata/data.jsonl"

In [6]:
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

In [7]:
def to_text(example):
    # Convert messages -> single string bằng chat template
    txt = tokenizer.apply_chat_template(
        example["message"],
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": txt}

raw = load_dataset("json", data_files=data_path, split="train")
ds = raw.map(to_text, remove_columns=raw.column_names)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

In [8]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype="bfloat16"
)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/99.6M [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/3.96G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [9]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

In [10]:
cfg = SFTConfig(
    output_dir="./qwen_lora_sft",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=1,
    dataset_text_field="text",
    max_length=256,
    logging_steps=10,
    save_steps=500,
    save_total_limit=2,
    bf16=True,
    packing=False,
    report_to="none"
)

In [11]:
trainer = SFTTrainer(
    model=model,
    train_dataset=ds,
    peft_config=lora_config,
    args=cfg
)

Adding EOS to train dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

In [12]:
trainer.train()
trainer.model.save_pretrained("./qwen_lora_adapter")
tokenizer.save_pretrained("./qwen_lora_adapter")

Step,Training Loss
10,2.213300
20,1.638800
30,1.465200
40,1.264700
50,1.104900
60,1.210000
70,0.859700
80,0.844800
90,0.941200
100,0.757300


('./qwen_lora_adapter/tokenizer_config.json',
 './qwen_lora_adapter/special_tokens_map.json',
 './qwen_lora_adapter/chat_template.jinja',
 './qwen_lora_adapter/vocab.json',
 './qwen_lora_adapter/merges.txt',
 './qwen_lora_adapter/added_tokens.json',
 './qwen_lora_adapter/tokenizer.json')

# Inference

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import torch

model_id = "Qwen/Qwen3-4B"
adapter_dir = "/kaggle/input/qwen-lora-adapter/qwen_lora_adapter"

tokenizer = AutoTokenizer.from_pretrained(adapter_dir)
tokenizer.pad_token = tokenizer.eos_token

base = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

# 3) attach LoRA adapter
model = PeftModel.from_pretrained(base, adapter_dir)
model.eval()

2025-08-10 07:26:27.583493: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1754810787.929352      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1754810788.039787      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/3.96G [00:00<?, ?B/s]

...

model-00003-of-00003.safetensors:   0%|          | 0.00/99.6M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/peft/config.py:165: UserWarning: Unexpected keyword arguments ['qalora_group_size', 'target_parameters', 'use_qalora'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


('/kaggle/working/qwen3_merged/tokenizer_config.json',
 '/kaggle/working/qwen3_merged/special_tokens_map.json',
 '/kaggle/working/qwen3_merged/chat_template.jinja',
 '/kaggle/working/qwen3_merged/vocab.json',
 '/kaggle/working/qwen3_merged/merges.txt',
 '/kaggle/working/qwen3_merged/added_tokens.json',
 '/kaggle/working/qwen3_merged/tokenizer.json')

In [3]:
# 4) build chat prompt from messages using the chat template
messages = [
    {"role": "system", "content": "Bạn là một AI tư vấn tuyển sinh đại học chuyên nghiệp. Hãy trả lời các câu hỏi một cách chính xác, hữu ích và thân thiện. Có thể sử dụng những thông tin được cung cấp để đưa ra câu trả lời hoặc lời khuyên tốt nhất."},
    {"role": "user", "content": f"""
Thông tin tham khảo:
Điểm chuẩn trường UET - Đại Học Công nghệ - ĐHQG Hà Nội năm 2024

Năm 2024, trường Đại học Công nghệ - Đại học Quốc gia Hà Nội tuyển sinh 2.960 chỉ tiêu theo 5 phương thức tuyển sinh: Xét tuyển thẳng, Ưu tiên xét tuyển; Xét tuyển kết quả thi tốt nghiệp THPT năm 2024; Xét tuyển kết quả thi Đánh giá năng lực do ĐHQGHN tổ chức; Xét tuyển theo chứng chỉ quốc tế gồm SAT, A-Level, ACT và Xét tuyển kết hợp kết quả thi tốt nghiệp THPT với chứng chỉ tiếng Anh quốc tế.

Điểm chuẩn UET - Đại học Công nghệ - ĐHQGHN năm 2024 dựa theo điểm thi tốt nghiệp THPT, điểm thi ĐGNL của ĐHQGHN, xét chứng chỉ quốc tế đã được công bố. Chi tiết được đăng tải bên dưới.

                              Tên ngành    Tổ hợp môn  Điểm chuẩn  Ghi chú
                    Công nghệ thông tin A00; A01; D01       27.80      NaN
                  Công nghệ nông nghiệp A00; A01; B00       22.50      NaN
     Kỹ thuật điều khiển và tự động hoá A00; A01; D01       27.05      NaN
                       Trí tuệ nhân tạo A00; A01; D01       27.12      NaN
                    Kỹ thuật năng lượng A00; A01; D01       24.59      NaN
                     Hệ thống thông tin A00; A01; D01       26.87      NaN
  Mạng máy tính và truyền thông dữ liệu A00; A01; D01       26.92      NaN
                         Kỹ thuật Robot A00; A01; D01       25.99      NaN
         Thiết kế công nghiệp và đồ họa A00; A01; D01       24.64      NaN
                      Kỹ thuật máy tính A00; A01; D01       26.97      NaN
                        Vật lý kỹ thuật A00; A01; D01       25.24      NaN
                            Cơ kỹ thuật A00; A01; D01       26.03      NaN
            Công nghệ kỹ thuật xây dựng A00; A01; D01       23.91      NaN
        Công nghệ kỹ thuật cơ – điện tử A00; A01; D01       26.27      NaN
            Công nghệ hàng không vũ trụ A00; A01; D01       24.61      NaN
                      Khoa học máy tính A00; A01; D01       27.58      NaN
Công nghệ kỹ thuật điện tử – viễn thông A00; A01; D01       26.30      NaN

Câu hỏi: Ngành AI tại UET lấy bao nhiêu điểm.
"""}
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
input_len = inputs.input_ids.shape[-1]

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=True,
        temperature=0.5,
        top_p=0.9,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )

new_tokens = outputs[0, input_len:]
answer = tokenizer.decode(new_tokens, skip_special_tokens=True, clean_up_tokenization_spaces=False).strip()
print(answer)

Ngành **Trí tuệ nhân tạo** tại Trường Đại học Công nghệ - Đại học Quốc gia Hà Nội năm 2024 có điểm chuẩn là **27,12**. Đây là ngành có điểm cao hơn nhiều ngành khác, phản ánh tính cạnh tranh và chất lượng đào tạo trong lĩnh vực AI tại trường.


# Deploy

In [1]:
! pip install vllm pyngrok
! pip install --upgrade triton

INFO: pip is looking at multiple versions of transformers to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.6/386.6 MB 4.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.0/169.0 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 87.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 73.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 821.2/821.2 MB 2.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
NGROK_KEY = user_secrets.get_secret("NGROK_KEY")

In [3]:
from pyngrok import ngrok
ngrok.set_auth_token(NGROK_KEY)

In [21]:
import subprocess, os, signal

BASE_MODEL = "Qwen/Qwen3-4B"
LORA_PATH  = "/kaggle/input/qwen-lora-adapter/qwen_lora_adapter"  # đổi path adapter
MAX_LEN    = "2048"

vllm_cmd = [
    "python", "-m", "vllm.entrypoints.openai.api_server",
    "--model", BASE_MODEL,
    "--host", "0.0.0.0", "--port", "8000",
    "--dtype", "float16",
    "--max-model-len", MAX_LEN,
    "--enable-lora",
    "--lora-modules", f"myft={LORA_PATH}",
    "--max-lora-rank", "16",
    # "--enforce-eager",
]

vllm_log = open("/kaggle/working/vllm.log", "w")
vllm_proc = subprocess.Popen(vllm_cmd, stdout=vllm_log, stderr=subprocess.STDOUT, preexec_fn=os.setsid)

print("vLLM started with PID", vllm_proc.pid)


vLLM started with PID 366


In [31]:
!tail -n 20 /kaggle/working/vllm.log

INFO 08-10 08:58:16 [launcher.py:37] Route: /v1/responses/{response_id}/cancel, Methods: POST
INFO 08-10 08:58:16 [launcher.py:37] Route: /v1/chat/completions, Methods: POST
INFO 08-10 08:58:16 [launcher.py:37] Route: /v1/completions, Methods: POST
INFO 08-10 08:58:16 [launcher.py:37] Route: /v1/embeddings, Methods: POST
INFO 08-10 08:58:16 [launcher.py:37] Route: /pooling, Methods: POST
INFO 08-10 08:58:16 [launcher.py:37] Route: /classify, Methods: POST
INFO 08-10 08:58:16 [launcher.py:37] Route: /score, Methods: POST
INFO 08-10 08:58:16 [launcher.py:37] Route: /v1/score, Methods: POST
INFO 08-10 08:58:16 [launcher.py:37] Route: /v1/audio/transcriptions, Methods: POST
INFO 08-10 08:58:16 [launcher.py:37] Route: /v1/audio/translations, Methods: POST
INFO 08-10 08:58:16 [launcher.py:37] Route: /rerank, Methods: POST
INFO 08-10 08:58:16 [launcher.py:37] Route: /v1/rerank, Methods: POST
INFO 08-10 08:58:16 [launcher.py:37] Route: /v2/rerank, Methods: POST
INFO 08-10 08:58:16 [launcher.py

In [32]:
tunnel = ngrok.connect(8000, "http")

print("Public URL:", tunnel.public_url)  # hoặc: str(tunnel)
print("Gọi API: POST", f"{tunnel.public_url}/v1/chat/completions")

Public URL: https://f4f9586f58b7.ngrok-free.app
Gọi API: POST https://f4f9586f58b7.ngrok-free.app/v1/chat/completions


In [48]:
messages = [
    {"role": "system", "content": "Bạn là một AI tư vấn tuyển sinh đại học chuyên nghiệp. Hãy trả lời các câu hỏi một cách chính xác, hữu ích và thân thiện. Có thể sử dụng những thông tin được cung cấp để đưa ra câu trả lời hoặc lời khuyên tốt nhất."},
    {"role": "user", "content": """Thông tin tham khảo:
    Tổng quan
Viện Trí tuệ nhân tạo được thành lập theo quyết định số 162/QĐ-TCCB, ngày 18/03/2022 của Hiệu trưởng Trường Đại học Công nghệ. Tên tiếng Anh: Institute for Artificial Intelligence (IAI). Viện Trí tuệ nhân tạo là đơn vị đào tạo, nghiên cứu, phát triển công nghệ thuộc Trường ĐHCN, hoạt động theo qui chế tổ chức và hoạt động của Viện do Hiệu trưởng Trường ĐHCN ban hành.

Sứ mệnh
Trường ĐHCN xác định sứ mạng của Viện Trí tuệ nhân tạo là “Đào tạo nguồn nhân lực công nghệ chất lượng cao, trình độ cao trong lĩnh vực trí tuệ nhân tạo và các lĩnh vực liên ngành; nghiên cứu phát triển và ứng dụng trí tuệ nhân tạo trong các lĩnh vực đem lại lợi ích xã hội, từ đó đóng góp tích cực vào sự phát triển của Trường ĐHCN, của ĐHQGHN cũng như sự phát triển nền kinh tế và xã hội tri thức của đất nước trong xu thế cuộc cách mạng công nghiệp lần thứ tư”.

Tầm nhìn
Trở thành đơn vị dẫn đầu trong cả nước về đào tạo nguồn nhân lực chất lượng cao ngành trí tuệ nhân tạo, nghiên cứu khoa học và chuyển giao công nghệ liên ngành.

Cơ cấu tổ chức
Văn phòng Viện
Hội đồng khoa học đào tạo
Phòng thí nghiệm Học máy
Phòng thí nghiệm Xử lý ngôn ngữ tự nhiên
Phòng thí nghiệm trọng điểm Hệ thống tích hợp thông minh
Đội ngũ cán bộ
STT	Họ tên	Chức danh	Lĩnh vực	Email
1	TS. Trần Quốc Long	Viện trưởng, Giảng viên chính	Học máy, xử lý hình ảnh	tqlong@vnu.edu.vn
 2	PGS.TS. Nguyễn Việt Hà	Giảng viên cao cấp	Trí tuệ nhân tạo, Học máy, Xử lý ngôn ngữ tự nhiên 	hanv@vnu.edu.vn 
 3	PGS.TS. Nguyễn Phương Thái	Trưởng phòng Thí nghiệm xử lý ngôn ngữ tự nhiên, Giảng viên cao cấp	Xử lý ngôn ngữ tự nhiên	thainp@vnu.edu.vn 
 4	TS. Bùi Ngọc Thăng	Trưởng phòng Thí nghiệm Hệ thống tri thức	Học máy	thangbn@vnu.edu.vn 
5	TS. Nguyễn Thị Ngọc Diệp 	Phó trưởng phòng Thí nghiệm Học máy	Học máy, xử lý hình ảnh	ngocdiep@vnu.edu.vn
6	TS. Trần Hồng Việt	Giảng viên chính	Xử lý ngôn ngữ tự nhiên	 thviet@vnu.edu.vn
7	TS. Triệu Hải Long	Giảng viên	Xử lý ngôn ngữ tự nhiên	thlong@vnu.edu.vn 
8	TS. Bùi Văn Vượng	Giảng viên	Hệ thống tri thức, Toán học	 bui.vuong@vnu.edu.vn
9	TS. Nguyễn Kiêm Hùng	Giảng viên	Các hệ thống tích hợp thông minh, Kiến trúc máy tính	kiemhung@vnu.edu.vn
10	TS. Đỗ Thái Dương	Giảng viên chính	Giải tích phức, Lý thuyết đa thế vị	dtduong@vnu.edu.vn
11	TS. Nguyễn Bích Vân	 Giảng viên chính	 Lý thuyết biểu diễn, đại số kết hợp, ứng dụng đại số trong phương trình vi phân	nbvan@vnu.edu.vn
12	ThS. Quách Công Hoàng	Giảng viên	Các hệ thống tích hợp thông minh, Kiến trúc máy tính	 hoangqc@vnu.edu.vn
13	ThS. Ngô Minh Hương	Giảng viên	Xử lý ngôn ngữ tự nhiên, ngôn ngữ học 	nmhuong@vnu.edu.vn 
14	ThS. Nguyễn Thị Thùy Linh	Giảng viên	 Khai phá quy trình, mô hình ngôn ngữ	linh.nguyen@vnu.edu.vn 
15	ThS. Nguyễn Văn Phi	Giảng viên	Học máy, xử lý hình ảnh	phinv@vnu.edu.vn 
16	CN. Đỗ Thu Uyên	Trợ giảng	Học máy, học tăng cường 	uyendt@vnu.edu.vn 
17	CN. Nguyễn Hải Toàn	Trợ giảng	Học máy	nguyenhaitoan@vnu.edu.vn 
18	CN. Nguyễn Tiến Đạt	Trợ giảng	 Học máy, Xử lý ngôn ngữ tự nhiên	datntien@vnu.edu.vn
19	CN. Lương Sơn Bá	Trợ giảng	Trí tuệ nhân tạo cho Y tế 	bals@vnu.edu.vn
20	CN. Phạm Tiến Du	Trợ giảng	 Học máy	ptdu@vnu.edu.vn
21	CN. Trịnh Ngọc Huỳnh	Trợ giảng	 Trí tuệ nhân tạo cho Y tế, mô hình sinh	huynhtn@vnu.edu.vn
22	CN. Nguyễn Khánh Ly	Chuyên viên Văn phòng Viện	 	lynk@vnu.edu.vn 
23	TS. Hoàng Thanh Tùng	Giảng viên kiêm nhiệm 	Học máy 	 
24	PGS.TS. Lê Anh Cường	Giảng viên kiêm nhiệm 	Học máy 	 
25	TS. Phạm Tiến Lâm	Giảng viên kiêm nhiệm 	Hệ thống tri thức, Khoa học Vật liệu	 
26	TS. Nguyễn Việt Cường	Giảng viên kiêm nhiệm 	Hệ thống máy tính hiệu năng cao	 
27	TS. Trần Tiến Hải	Giảng viên kiêm nhiệm	 	 
28	TS. Trần Văn Khánh	Giảng viên kiêm nhiệm	 	 


Câu hỏi: Viện trí tuệ nhân tạo UET có bao nhiêu giảng viên là Tiến sĩ"""
}
]

In [49]:
from openai import OpenAI

openai_api_key = "EMPTY"
openai_api_base = "https://f4f9586f58b7.ngrok-free.app/v1"

client = OpenAI(
    api_key=openai_api_key,
    base_url=openai_api_base,
)

chat_response = client.chat.completions.create(
    model="Qwen/Qwen3-4B",
    messages=messages,
    temperature=0.5,
    top_p=0.9,
    extra_body={
        "chat_template_kwargs": {"enable_thinking": False},
    },
)
print(chat_response.choices[0].message.content)

Theo thông tin được cung cấp, **Viện Trí tuệ Nhân tạo (IAI)** của Trường Đại học Công nghệ (UET) có **25 giảng viên là Tiến sĩ**. Dưới đây là danh sách các giảng viên có chức danh **Tiến sĩ**:

1. **TS. Trần Quốc Long** – Viện trưởng, Giảng viên chính  
2. **PGS.TS. Nguyễn Việt Hà** – Giảng viên cao cấp  
3. **PGS.TS. Nguyễn Phương Thái** – Trưởng phòng Thí nghiệm xử lý ngôn ngữ tự nhiên, Giảng viên cao cấp  
4. **TS. Bùi Ngọc Thăng** – Trưởng phòng Thí nghiệm Hệ thống tri thức  
5. **TS. Nguyễn Thị Ngọc Diệp** – Phó trưởng phòng Thí nghiệm Học máy  
6. **TS. Trần Hồng Việt** – Giảng viên chính  
7. **TS. Triệu Hải Long** – Giảng viên  
8. **TS. Bùi Văn Vượng** – Giảng viên  
9. **TS. Nguyễn Kiêm Hùng** – Giảng viên chính  
10. **TS. Đỗ Thái Dương** – Giảng viên chính  
11. **TS. Nguyễn Bích Vân** – Giảng viên chính  
12. **TS. Hoàng Thanh Tùng** – Giảng viên kiêm nhiệm  
13. **PGS.TS. Lê Anh Cường** – Giảng viên kiêm nhiệm  
14. **TS. Phạm Tiến Lâm** – Giảng viên kiêm nhiệm  
15. **TS

In [50]:
!ps -ef | grep vllm

root         366      36  1 08:55 ?        00:00:15 python3 -m vllm.entrypoints.openai.api_server --model Qwen/Qwen3-4B --host 0.0.0.0 --port 8000 --dtype float16 --max-model-len 2048 --enable-lora --lora-modules myft=/kaggle/input/qwen-lora-adapter/qwen_lora_adapter --max-lora-rank 16
root         803      36  0 09:11 pts/0    00:00:00 /bin/bash -c ps -ef | grep vllm
root         805     803  0 09:11 pts/0    00:00:00 grep vllm


In [51]:
!ps -ef | grep ngrok

root         783      36  0 08:58 ?        00:00:01 /root/.config/ngrok/ngrok start --none --log stdout
root         806      36  0 09:11 pts/0    00:00:00 /bin/bash -c ps -ef | grep ngrok
root         808     806  0 09:11 pts/0    00:00:00 grep ngrok
